# 5-DPO

对应 `trainer/train_dpo.py`。DPO 是 RLHF 里不需要在线 reward model 的偏好优化：chosen 相对 rejected 的 log 比，再减去参考模型的同一差值。

$$\mathcal{L}_{\mathrm{DPO}} = -\log\sigma\big(\beta[(\log\pi_\theta(y_w|x)-\log\pi_\theta(y_l|x))-(\log\pi_{\mathrm{ref}}(y_w|x)-\log\pi_{\mathrm{ref}}(y_l|x))]\big)$$


In [ ]:
import os, sys, math, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
import torch
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("../model")
print("device:", device, "vocab:", tokenizer.vocab_size)
def tiny_model(use_moe=False):
    cfg = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=use_moe)
    return MiniMindForCausalLM(cfg).to(device), cfg

import torch.nn.functional as F


In [ ]:
def logits_to_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=2)
    return torch.gather(log_probs, 2, labels.unsqueeze(2)).squeeze(-1)

def dpo_loss(ref_log_probs, policy_log_probs, mask, beta=0.1):
    ref_log_probs = (ref_log_probs * mask).sum(dim=1)
    policy_log_probs = (policy_log_probs * mask).sum(dim=1)
    bsz = ref_log_probs.shape[0]
    chosen_ref, reject_ref = ref_log_probs[:bsz // 2], ref_log_probs[bsz // 2:]
    chosen_pi, reject_pi = policy_log_probs[:bsz // 2], policy_log_probs[bsz // 2:]
    logits = (chosen_pi - reject_pi) - (chosen_ref - reject_ref)
    return -F.logsigmoid(beta * logits).mean()


参考模型冻结，只更新 policy。正式训练会把 chosen/rejected 在 batch 维拼在一起。


In [ ]:
policy, cfg = tiny_model()
ref, _ = tiny_model()
ref.load_state_dict(policy.state_dict())
ref.eval()
for p in ref.parameters():
    p.requires_grad_(False)

ds = DPODataset("./toydata/dpo_data.jsonl", tokenizer, max_length=96)
loader = DataLoader(ds, batch_size=1)
optimizer = optim.AdamW(policy.parameters(), lr=1e-4)
policy.train()

for step, batch in enumerate(loader, start=1):
    x = torch.cat([batch["x_chosen"], batch["x_rejected"]], 0).to(device)
    y = torch.cat([batch["y_chosen"], batch["y_rejected"]], 0).to(device)
    mask = torch.cat([batch["mask_chosen"], batch["mask_rejected"]], 0).to(device)
    with torch.no_grad():
        ref_lp = logits_to_log_probs(ref(x).logits, y)
    out = policy(x)
    loss = dpo_loss(ref_lp, logits_to_log_probs(out.logits, y), mask) + out.aux_loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(f"step {step} dpo_loss={float(loss):.4f}")


完整训练：`cd trainer && python train_dpo.py`，权重 `../out/rlhf_{hidden_size}.pth`。DPO 更常改善礼貌/偏好，对“智力”提升有限。
